In [ ]:
import os
import json
import time
import random
import logging
from datasets import load_dataset, Dataset, DatasetDict
from translatepy import Translator
from translatepy.translators import YandexTranslate
from tqdm.auto import tqdm

# Создаём переводчик
yandex = YandexTranslate()

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# Тест переводчика
test_text = "Hello, world!"
result = yandex.translate(test_text, "ru")
print(f"Original: {test_text}")
print(f"Translated: {result}")

In [3]:
SOURCE_REPO_ID = "DeepPavlov/Mantis"
LOCAL_SAVE_PATH = "./Mantis_ru"

# Файлы прогресса для каждого сплита
PROGRESS_FILES = {
    'train': "translated_mantis_train_progress.jsonl",
    'dev': "translated_mantis_dev_progress.jsonl",
    'test': "translated_mantis_test_progress.jsonl"
}
CACHE_FILE = "translation_mantis_cache.jsonl"

In [4]:
def load_cache() -> dict[str, str]:
    cache: dict[str, str] = {}
    if not os.path.exists(CACHE_FILE):
        return cache
    try:
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj: dict[str, str] = json.loads(line)
                    cache[obj["text"]] = obj["translation"]
                except Exception:
                    continue
    except Exception as e:
        logging.error(f"Failed to load cache: {e}")
    return cache

def append_cache(text: str, translation: str) -> None:
    try:
        with open(CACHE_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps({"text": text, "translation": translation}, ensure_ascii=False) + "\n")
    except Exception as e:
        logging.error(f"Failed to append cache: {e}")

translation_cache: dict[str, str] = load_cache()

In [9]:
def translate_with_yandex(text: str, retries: int = 3, delay: int = 5) -> tuple[str, bool]:
    """
    Возвращает (перевод, успех_или_нет)
    """
    if not isinstance(text, str) or text.strip() == "":
        return "", True
    
    if text in translation_cache:
        return translation_cache[text], True
    
    for attempt in range(retries):
        try:
            time.sleep(0.5)
            result = yandex.translate(text, "ru")
            
            # Извлекаем текст из результата
            if hasattr(result, 'result'):
                translated_text = str(result.result)
            else:
                translated_text = str(result)
            
            translation_cache[text] = translated_text
            append_cache(text, translated_text)
            return translated_text, True
            
        except Exception as e:
            logging.warning(f"Translation error for text: '{text[:50]}...'. Attempt {attempt + 1}/{retries}. Error: {e}")
            if attempt < retries - 1:
                time.sleep(delay)
    
    # Если все попытки не удались
    logging.error(f"Failed to translate after {retries} attempts: '{text[:50]}...'")
    return "", False

def translate_dialog(dialog: list) -> tuple[list, bool]:
    """
    Переводит диалог. Возвращает список с переведёнными сообщениями,
    сохраняя исходную структуру (только поле message заменяется на перевод).
    """
    translated_dialog = []
    all_success = True
    
    for turn in dialog:
        # Создаём копию с сохранением всех полей
        translated_turn = turn.copy()
        
        # Переводим только текст сообщения
        if 'message' in turn and isinstance(turn['message'], str):
            translated_text, success = translate_with_yandex(turn['message'])
            translated_turn['message'] = translated_text  # ЗАМЕНЯЕМ, а не добавляем поле
            if not success:
                all_success = False
        else:
            translated_turn['message'] = turn.get('message', '')
            all_success = False
        
        translated_dialog.append(translated_turn)
    
    return translated_dialog, all_success

In [10]:
def process_and_translate_split(split_name, source_dataset_split, progress_file, text_fields=['title']):
    """
    Функция для обработки одного сплита датасета Mantis.
    Переводит поле 'dialog' (каждое сообщение) и поля, указанные в text_fields (например, 'title').
    Структура переведённого диалога идентична оригинальной (message заменяется на перевод).
    """
    translated_records = []
    failed_indices = []
    
    # Загружаем уже переведенные записи
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    record = json.loads(line)
                    if record.get('_failed', False):
                        failed_indices.append(record.get('_index', -1))
                    else:
                        translated_records.append(record)
                except:
                    continue
        logging.info(f"Resuming {split_name}. Found {len(translated_records)} successful, {len(failed_indices)} failed records.")
    
    # Создаем множество индексов уже успешно переведенных
    successful_indices = {record['_index'] for record in translated_records if '_index' in record}
    
    start_index = len(translated_records) + len(failed_indices)
    total_records = len(source_dataset_split)
    
    if start_index < total_records:
        logging.info(f"Starting translation for '{split_name}' from index {start_index}...")
        
        with open(progress_file, "a", encoding="utf-8") as f:
            pbar = tqdm(
                enumerate(source_dataset_split.select(range(start_index, total_records))),
                desc=f"Translating {split_name}",
                total=total_records - start_index
            )
            
            for idx, example in pbar:
                global_idx = start_index + idx
                
                # Проверяем, не переведен ли уже этот индекс
                if global_idx in successful_indices:
                    continue
                
                # Переводим диалог
                original_dialog = example['dialog']
                translated_dialog, dialog_success = translate_dialog(original_dialog)
                
                # Создаём запись
                new_record = {
                    '_index': global_idx,
                    '_failed': not dialog_success,
                    'dialog': original_dialog,           # оригинал
                    'dialog_ru': translated_dialog,      # перевод (идентичная структура)
                }
                
                # Копируем остальные поля и переводим текстовые
                other_success = True
                for key, value in example.items():
                    if key == 'dialog':
                        continue
                    
                    # Если поле нужно перевести и это строка
                    if key in text_fields and isinstance(value, str):
                        translated_value, success = translate_with_yandex(value)
                        new_record[f"{key}_ru"] = translated_value  # добавляем поле с переводом
                        new_record[key] = value                      # сохраняем оригинал
                        if not success:
                            other_success = False
                    else:
                        # Остальные поля копируем без изменений
                        new_record[key] = value
                
                # Если какие-то из дополнительных текстовых полей не перевелись, запись считается неудачной
                new_record['_failed'] = new_record['_failed'] or not other_success
                
                f.write(json.dumps(new_record, ensure_ascii=False) + "\n")
                f.flush()  # Немедленно сохраняем на диск
                
                if not new_record['_failed']:
                    translated_records.append(new_record)
                else:
                    failed_indices.append(global_idx)
    
    # Загружаем все записи для финального датасета (только успешные)
    all_successful = []
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    record = json.loads(line)
                    if not record.get('_failed', False):
                        # Удаляем временные поля
                        record.pop('_index', None)
                        record.pop('_failed', None)
                        all_successful.append(record)
                except:
                    continue
    
    if not all_successful:
        logging.error(f"No records were successfully translated for {split_name}. Aborting.")
        return None
    
    # Выводим статистику
    total = len(source_dataset_split)
    successful_count = len(all_successful)
    failed_count = total - successful_count
    logging.info(f"{split_name}: {successful_count}/{total} translated successfully ({failed_count} failed)")
    
    if failed_count > 0:
        logging.info(f"To retry failed translations, delete or modify {progress_file} and run again")
    
    return Dataset.from_list(all_successful)

In [ ]:
# Загружаем датасет Mantis
logging.info(f"Loading source dataset '{SOURCE_REPO_ID}'...")
source_dataset = load_dataset(SOURCE_REPO_ID)
print("Source dataset loaded:")
print(source_dataset)

# Проверяем структуру первого примера из train
print("\nExample structure from train:")
print(source_dataset['train'][0])
print("\nExample structure from dev:")
print(source_dataset['dev'][0])
print("\nExample structure from test:")
print(source_dataset['test'][0])

In [ ]:
# Переводим все сплиты
translated_splits = {}
for split in ['train', 'dev', 'test']:
    progress_file = PROGRESS_FILES[split]
    translated_splits[split] = process_and_translate_split(
        f"{split}", 
        source_dataset[split], 
        progress_file, 
        text_fields=['title']
    )
    
    if translated_splits[split] is None:
        logging.error(f"Failed to translate {split} split")
        break

In [ ]:
# Если все сплиты переведены успешно, создаем DatasetDict и сохраняем
if all(v is not None for v in translated_splits.values()):
    final_dataset = DatasetDict(translated_splits)
    
    print("\n" + "=" * 60)
    print("FINAL TRANSLATED DATASET")
    print("=" * 60)
    for split_name, ds in final_dataset.items():
        print(f"\n{split_name.upper()}:")
        print(ds)
    
    print("\nExample from translated train split (first record):")
    example = final_dataset['train'][0]
    print(f"Original title: {example.get('title', 'N/A')}")
    print(f"Translated title: {example.get('title_ru', 'N/A')}")
    print(f"Original category: {example.get('category', 'N/A')}")
    print(f"Original dialog_time: {example.get('dialog_time', 'N/A')}")
    if example.get('dialog') and len(example['dialog']) > 0:
        print(f"\nOriginal dialog (first turn): {example['dialog'][0].get('message', 'N/A')}")
    if example.get('dialog_ru') and len(example['dialog_ru']) > 0:
        print(f"Translated dialog (first turn): {example['dialog_ru'][0].get('message_ru', 'N/A')}")
    
    logging.info(f"Saving translated dataset locally to '{LOCAL_SAVE_PATH}'...")
    final_dataset.save_to_disk(LOCAL_SAVE_PATH)
    print(f"\nDataset saved to {LOCAL_SAVE_PATH}")
else:
    logging.error("One of the splits failed to process. Halting.")

In [ ]:
# Функция для повторной попытки перевода неудачных записей
def retry_failed_translations(split_name, source_dataset_split, progress_file, text_fields=['title']):
    """
    Перезапускает только те записи, которые не перевелись в прошлый раз
    """
    failed_records = []
    
    if not os.path.exists(progress_file):
        logging.error(f"Progress file {progress_file} not found")
        return None
    
    # Загружаем неудачные записи
    with open(progress_file, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            if record.get('_failed', False):
                failed_records.append(record)
    
    if not failed_records:
        logging.info(f"No failed records found for {split_name}")
        return None
    
    logging.info(f"Retrying {len(failed_records)} failed translations for {split_name}...")
    
    # Создаем временный файл для новых успешных переводов
    temp_progress_file = progress_file + ".retry"
    successful_retries = 0
    
    with open(temp_progress_file, "w", encoding="utf-8") as f_out:
        # Сначала записываем все успешные записи из исходного файла
        with open(progress_file, "r", encoding="utf-8") as f_in:
            for line in f_in:
                record = json.loads(line)
                if not record.get('_failed', False):
                    f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
        
        # Пробуем перевести неудачные записи заново
        for record in tqdm(failed_records, desc=f"Retrying {split_name}"):
            original_dialog = record['dialog']
            translated_dialog, dialog_success = translate_dialog(original_dialog)
            
            other_success = True
            new_record = {
                '_index': record['_index'],
                '_failed': not dialog_success,
                'dialog': original_dialog,
                'dialog_ru': translated_dialog,
            }
            
            # Копируем остальные поля
            for key, value in record.items():
                if key in ['_index', '_failed', 'dialog', 'dialog_ru']:
                    continue
                
                # Если поле нужно перевести и это строка
                if key in text_fields and isinstance(value, str):
                    translated_value, success = translate_with_yandex(value)
                    new_record[f"{key}_ru"] = translated_value
                    new_record[key] = value
                    if not success:
                        other_success = False
                else:
                    new_record[key] = value
            
            new_record['_failed'] = new_record['_failed'] or not other_success
            
            if not new_record['_failed']:
                successful_retries += 1
            
            f_out.write(json.dumps(new_record, ensure_ascii=False) + "\n")
    
    # Заменяем старый файл новым
    os.replace(temp_progress_file, progress_file)
    
    logging.info(f"Retry complete for {split_name}: {successful_retries}/{len(failed_records)} succeeded")
    
    # Возвращаем обновленный датасет
    return process_and_translate_split(split_name, source_dataset_split, progress_file, text_fields)

In [ ]:
def check_translation_status():
    """
    Проверяет статус всех переводов
    """
    print("=" * 60)
    print("TRANSLATION STATUS REPORT")
    print("=" * 60)
    
    for split in ['train', 'dev', 'test']:
        progress_file = PROGRESS_FILES[split]
        if os.path.exists(progress_file):
            successful = 0
            failed = 0
            with open(progress_file, "r", encoding="utf-8") as f:
                for line in f:
                    record = json.loads(line)
                    if record.get('_failed', False):
                        failed += 1
                    else:
                        successful += 1
            
            total = successful + failed
            progress = (successful / total * 100) if total > 0 else 0
            print(f"\n{split.upper()}: {successful}/{total} ({progress:.1f}%) - {failed} failed")
        else:
            print(f"\n{split.upper()}: No progress file found")
    
    print("\n" + "=" * 60)

# Проверяем статус
check_translation_status()

In [ ]:
# Загрузка переведённого датасета DeepPavlov/Mantis-ru на Hugging Face Hub

from datasets import load_from_disk, DatasetDict
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")

# Путь к сохранённому датасету
LOCAL_PATH = "/Users/polinakremneva/code/deep-pavlov/Mantis_ru"

# Название репозитория на Hub
REPO_ID = "DeepPavlov/Mantis_ru"

# Загружаем датасет из локальной папки
print(f"Loading dataset from {LOCAL_PATH}...")
dataset = load_from_disk(LOCAL_PATH)

print("\nDataset loaded successfully!")
print(dataset)

# %%
# Проверяем структуру перед загрузкой
print("\n" + "="*60)
print("DATASET STRUCTURE VERIFICATION")
print("="*60)

for split_name, ds in dataset.items():
    print(f"\n{split_name.upper()} split:")
    print(f"  Number of examples: {len(ds)}")
    print(f"  Features: {ds.column_names}")
    
    # Проверяем структуру dialog_ru
    example = ds[0]
    print(f"  Example keys: {list(example.keys())}")
    if 'dialog_ru' in example:
        print(f"  dialog_ru[0] keys: {list(example['dialog_ru'][0].keys())}")
        print(f"  First message (ru): {example['dialog_ru'][0].get('message', 'N/A')[:80]}...")

# %%
# Загружаем датасет на Hugging Face Hub
print(f"\n{'='*60}")
print(f"Uploading dataset to Hugging Face Hub...")
print(f"Repository: {REPO_ID}")
print(f"{'='*60}")

# Для датасета Mantis он уже имеет три сплита, поэтому загружаем целиком
dataset.push_to_hub(
    REPO_ID,
    private=False,  # Можно поставить True для приватного репозитория
    commit_message="Initial upload of translated Mantis dataset (EN -> RU)",
)

print(f"\nDone! Dataset available at: https://huggingface.co/datasets/{REPO_ID}")